In [ ]:
import json, os, sys, glob, subprocess, time, torch
from pathlib import Path

# device smoke
assert torch.cuda.device_count()==2, "Expected 2x T4"
print("devices",[torch.cuda.get_device_name(i) for i in range(2)],flush=True)

# source
cands=[d for d in glob.glob("/kaggle/input/*") if os.path.isdir(d)]
repo=next((d for d in cands if (Path(d)/"pyproject.toml").exists() and "loopkit" in os.path.basename(d).lower()),None)
if repo is None:
    repo="/kaggle/working/loopkit"
    if not (Path(repo)/"pyproject.toml").exists():
        subprocess.run(["git","clone","-q","https://github.com/10xdev4u-alt/loopkit.git",repo],check=True)
sys.path.insert(0,str(Path(repo)/"src"))
print("repo",repo,flush=True)

import numpy as np
from loopkit.geometry import joint_itq, fp4_itq, spectral_break_even, lambda_law
from loopkit.nvfp4 import quantize_nvfp4, dequantize_nvfp4

rng=np.random.default_rng(20261123)
n,d=512,2304
k=np.arange(1,d+1,dtype=np.float64)
sig=np.maximum(k**(-0.27),1e-4)
U,_=np.linalg.qr(rng.standard_normal((n,n)))
V,_=np.linalg.qr(rng.standard_normal((d,d)))
Z=(U[:,:d] if False else U) @ np.diag(sig[:min(n,d)]) @ V[:min(n,d),:] if False else None
# build n x d with the intended spectrum
Z=(rng.standard_normal((n,d))).astype(np.float64)
s=np.linalg.svd(Z,compute_uv=False)
Z=Z*(sig[:len(s)]/s)[None,:] if len(s)==d else Z
# inject d2027-style outlier: 16x energy at column 2027
Z[:,2027]*=16.0

def lam(a):
    a=np.asarray(a,dtype=np.float64)
    return 1.0-(np.abs(a).sum(axis=1)/np.linalg.norm(a,axis=1))**2/a.shape[1]

def bimodality(a):
    a=np.asarray(a,dtype=np.float64).ravel(); a=(a-a.mean())/(a.std()+1e-12)
    g1=np.mean(a**3); g2=np.mean(a**4)-3
    return float((g1**2+1)/(g2+3*(len(a)-1)**2/((len(a)-2)*(len(a)-3))))

t0=time.perf_counter()
out={"shape":[int(n),int(d)],"outlier_dim":2027}
out["lambda_svd_mean"]=float(lam(Z).mean())
out["lambda_svd_max"]=float(lam(Z).max())
out["bc_svd"]=bimodality(Z)

R=joint_itq([Z],iters=50,seed=0)
ZR=Z@R
out["lambda_itq_mean"]=float(lam(ZR).mean())
out["lambda_itq_max"]=float(lam(ZR).max())
out["bc_itq"]=bimodality(ZR)
out["rotation_orthogonal"]=bool(np.allclose(R.T@R,np.eye(d),atol=1e-6))
out["itq_seconds"]=round(time.perf_counter()-t0,2)

# FP4 gate: same harness, binary target vs FP4-aware target
t1=time.perf_counter()
base=dequantize_nvfp4(quantize_nvfp4(Z.astype(np.float32))).astype(np.float64)
mse_base=float(np.mean((Z-base)**2))
R4,_=fp4_itq(Z,iters=20,seed=0)
Z4=Z@R4
rq=dequantize_nvfp4(quantize_nvfp4(Z4.astype(np.float32))).astype(np.float64)
mse_rot=float(np.mean((Z4-rq)**2))
out["fp4_mse_baseline"]=mse_base
out["fp4_mse_rotated"]=mse_rot
out["fp4_relative_gain"]=float((mse_base-mse_rot)/max(mse_base,1e-12))
out["fp4_seconds"]=round(time.perf_counter()-t1,2)

out["gamma_star"]={"svd":spectral_break_even(0.54),"rot":spectral_break_even(0.393),"itq":spectral_break_even(0.30)}
gates={
 "geometry_replication": out["gamma_star"]["itq"]>out["gamma_star"]["svd"]+0.04,
 "rotation_orthogonal": out["rotation_orthogonal"],
 "lambda_drop": out["lambda_itq_mean"] < out["lambda_svd_mean"],
 "fp4_gain": out["fp4_relative_gain"] > 0.0,
}
out["gates"]=gates
out["verdict"]="PASS" if all(gates.values()) else "NEGATIVE"
json.dump(out,open("/kaggle/working/geometry_gate.json","w"),indent=2)
print("GEOMETRY GATE",out["verdict"],json.dumps(gates),flush=True)
print("metrics",{k:round(v,5) for k,v in out.items() if isinstance(v,float)},flush=True)
